# Calcul des effets marginaux des variables macroéconomiques après le GVAR

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# -----------------------------
# 1) Assemblage des matrices du GVAR
# -----------------------------

def build_gvar_matrices(W, phi, beta, gamma):
    """
    Construit G0, {Gp}, {Gamma_l} à partir des coeffs sectoriels.
    
    Notation:
      Z*_i(t-p) = sum_{j != i} w_ij Z_j(t-p)
      Equation i: Z_i(t) = sum_p phi_{i,p} Z_i(t-p) + sum_p beta_{i,p} Z*_i(t-p) + sum_{k,l} gamma_{i,k,l} X_k(t-l) + eps
    
    Inputs
    - W: (m,m)
    - phi: (m,P)  -> phi[:,p-1] correspond à lag p (p=1..P). Si P=0 => array vide.
    - beta: (m,Pstar+1) -> beta[:,0] = terme contemporain sur Z* (p=0)
    - gamma: (m,K,L+1)
    
    Outputs
    - G0: (m,m)
    - G_list: liste [G1,...,GP_eff] où GP_eff = max(P, Pstar)
    - Gamma_list: liste [Gamma_0,...,Gamma_L], chaque (m,K)
    - B0: (m,m) matrice "spatiale contemporaine" telle que G0 = I - B0
    """
    m = W.shape[0]
    P = phi.shape[1] if phi.size else 0
    Pstar = beta.shape[1] - 1
    K = gamma.shape[1]
    L = gamma.shape[2] - 1
    
    # B0 : effet contemporain via beta[:,0] * W
    B0 = (beta[:, [0]] * W)  # (m,1)*(m,m) -> broadcast (m,m)
    np.fill_diagonal(B0, 0.0)  # par sécurité
    
    G0 = np.eye(m) - B0
    
    # Gp pour p=1..max(P,Pstar)
    P_eff = max(P, Pstar)
    G_list = []
    for p in range(1, P_eff + 1):
        Gp = np.zeros((m, m))
        
        # retards propres: sur la diagonale
        if p <= P:
            np.fill_diagonal(Gp, phi[:, p-1])
        
        # retards étrangers: beta[:,p] * W sur hors-diagonale
        if p <= Pstar:
            Gp += (beta[:, [p]] * W)
            np.fill_diagonal(Gp, np.diag(Gp) + 0.0)  # ne change rien, juste explicite
        
        G_list.append(Gp)
    
    # Gamma_l : (m,K)
    Gamma_list = []
    for l in range(0, L + 1):
        Gamma_l = gamma[:, :, l]  # (m,K)
        Gamma_list.append(Gamma_l)
    
    return G0, G_list, Gamma_list, B0


def reduced_form_matrices(G0, G_list, Gamma_list):
    """
    Calcule la forme réduite:
      Z_t = sum_p A_p Z_{t-p} + sum_l C_l X_{t-l} + e_t
    avec A_p = G0^{-1} G_p et C_l = G0^{-1} Gamma_l
    """
    G0_inv = np.linalg.inv(G0)
    A_list = [G0_inv @ Gp for Gp in G_list]
    C_list = [G0_inv @ Gamma_l for Gamma_l in Gamma_list]
    return G0_inv, A_list, C_list


# -----------------------------
# 2) Effets marginaux (statique + décomposition spatiale)
# -----------------------------

def marginal_effects_total(G0_inv, Gamma_list):
    """
    Effet marginal total de X_{t-l} sur Z_t:
      ME_l = G0^{-1} Gamma_l   (m,K)
    """
    return [G0_inv @ Gamma_l for Gamma_l in Gamma_list]


def marginal_effects_spatial_decomp(B0, Gamma_l, k_max=10, tol=1e-10):
    """
    Décomposition spatiale:
      (I - B0)^{-1} Gamma_l = sum_{k>=0} B0^k Gamma_l
    
    Retourne une liste [ME^(0), ME^(1), ..., ME^(k)] où ME^(k)=B0^k Gamma_l.
    Arrêt si la contribution devient négligeable.
    """
    m, K = Gamma_l.shape
    out = []
    Bk = np.eye(m)
    for k in range(0, k_max + 1):
        term = Bk @ Gamma_l
        out.append(term)
        
        # critère d'arrêt
        if np.max(np.abs(term)) < tol and k > 0:
            break
        
        Bk = Bk @ B0
    return out


# -----------------------------
# 3) Simulation dynamique pour un scénario macro
# -----------------------------

def simulate_deltaZ(A_list, C_list, dX: pd.DataFrame):
    """
    Simule la réponse de Z à un chemin macro exogène (différence scénario - baseline).
    ΔZ_t = sum_p A_p ΔZ_{t-p} + sum_l C_l ΔX_{t-l}
    
    Inputs:
    - A_list: [A1..AP_eff] matrices (m,m)
    - C_list: [C0..CL] matrices (m,K)
    - dX: DataFrame (T,K), index temps, colonnes macros (dans le même ordre que C_list)
    
    Output:
    - dZ: DataFrame (T,m)
    """
    T, K = dX.shape
    m = A_list[0].shape[0] if len(A_list) else C_list[0].shape[0]
    P_eff = len(A_list)
    L = len(C_list) - 1
    
    dZ = np.zeros((T, m))
    X = dX.values
    
    for t in range(T):
        # partie autoregressive
        ar_part = np.zeros(m)
        for p in range(1, P_eff + 1):
            if t - p >= 0:
                ar_part += A_list[p-1] @ dZ[t-p]
        
        # partie exogène
        exo_part = np.zeros(m)
        for l in range(0, L + 1):
            if t - l >= 0:
                exo_part += C_list[l] @ X[t-l]
        
        dZ[t] = ar_part + exo_part
    
    return pd.DataFrame(dZ, index=dX.index, columns=[f"Z_{i}" for i in range(m)])


def contributions_by_macro(A_list, C_list, dX: pd.DataFrame):
    """
    Décompose ΔZ en contributions par variable macro (additif),
    en simulant séparément chaque colonne de dX.
    """
    contrib = {}
    for k in range(dX.shape[1]):
        dXk = dX.copy()
        cols = list(dX.columns)
        for kk in range(dX.shape[1]):
            if kk != k:
                dXk.iloc[:, kk] = 0.0
        contrib[cols[k]] = simulate_deltaZ(A_list, C_list, dXk)
    return contrib


# -----------------------------
# 4) Graphiques utiles
# -----------------------------

def plot_heatmap_ME(ME_l, sectors, macro_vars, title):
    """
    Heatmap simple (matplotlib only).
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(ME_l, aspect='auto')
    ax.set_yticks(np.arange(len(sectors)))
    ax.set_yticklabels(sectors)
    ax.set_xticks(np.arange(len(macro_vars)))
    ax.set_xticklabels(macro_vars, rotation=45, ha='right')
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()


def plot_spatial_decomp_for_sector(me_terms, sector_name, sectors, macro_var, macro_vars, cumulative=True):
    """
    Plot de la propagation spatiale k=0..K pour un secteur donné et une macro.
    me_terms: liste de matrices (m,K) = [B0^k Gamma_l]
    """
    i = sectors.index(sector_name)
    j = macro_vars.index(macro_var)
    vals = np.array([term[i, j] for term in me_terms])
    if cumulative:
        vals = np.cumsum(vals)
        ylabel = "Effet marginal cumulé"
    else:
        ylabel = "Contribution par voisinage k"
    
    plt.figure(figsize=(9, 4))
    plt.plot(np.arange(len(vals)), vals)
    plt.title(f"Décomposition spatiale (macro={macro_var}) sur {sector_name}")
    plt.xlabel("Ordre de voisinage k")
    plt.ylabel(ylabel)
    plt.grid(False)
    plt.tight_layout()
    plt.show()


def plot_scenario_paths(dZ_by_scenario, sectors, top_n=6, title="ΔZ par scénario (top secteurs)"):
    """
    dZ_by_scenario: dict {scenario_name: DataFrame(T,m)}
    Sélectionne les top secteurs selon max abs sur le 1er scénario.
    """
    first_key = list(dZ_by_scenario.keys())[0]
    ref = dZ_by_scenario[first_key].values
    idx = np.argsort(np.max(np.abs(ref), axis=0))[::-1][:top_n]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    for scen, df in dZ_by_scenario.items():
        # moyenne des top secteurs (lisible) — tu peux aussi faire 1 courbe par secteur si tu préfères
        ax.plot(df.index, df.iloc[:, idx].mean(axis=1), label=scen)
    ax.set_title(title + " (moyenne des top secteurs)")
    ax.set_xlabel("Temps")
    ax.set_ylabel("ΔZ")
    ax.legend()
    ax.grid(False)
    plt.tight_layout()
    plt.show()


def plot_macro_contrib_for_sector(contrib_dict, sector_name, sectors, title="Contributions macro (secteur)"):
    """
    contrib_dict: dict macro_var -> DataFrame(T,m) contribution ΔZ due à cette macro
    """
    i = sectors.index(sector_name)
    df = pd.DataFrame({k: v.iloc[:, i] for k, v in contrib_dict.items()})
    
    plt.figure(figsize=(10, 5))
    plt.stackplot(df.index, df.T.values, labels=df.columns)
    plt.title(f"{title} : {sector_name}")
    plt.xlabel("Temps")
    plt.ylabel("ΔZ (décomposé)")
    plt.legend(loc="upper left", ncol=2)
    plt.grid(False)
    plt.tight_layout()
    plt.show()


# -----------------------------
# 5) Pipeline complet (exemple d'utilisation)
# -----------------------------

def run_pipeline(W, phi, beta, gamma, sectors, macro_vars, X_base, X_scenarios: dict,
                 spatial_kmax=12, show_lag=0, show_sector=None, show_macro=None):
    """
    W, phi, beta, gamma -> matrices
    X_base: DataFrame(T,K)
    X_scenarios: dict name->DataFrame(T,K)
    """
    # 1) Assemble
    G0, G_list, Gamma_list, B0 = build_gvar_matrices(W, phi, beta, gamma)
    G0_inv, A_list, C_list = reduced_form_matrices(G0, G_list, Gamma_list)
    
    # 2) Effets marginaux (total)
    ME_list = marginal_effects_total(G0_inv, Gamma_list)  # list of (m,K)
    ME_l = ME_list[show_lag]
    plot_heatmap_ME(ME_l, sectors, macro_vars, title=f"Effet marginal total ME_l (lag l={show_lag})")
    
    # 3) Décomposition spatiale pour un lag l (k=0..)
    me_terms = marginal_effects_spatial_decomp(B0, Gamma_list[show_lag], k_max=spatial_kmax)
    
    if show_sector is not None and show_macro is not None:
        plot_spatial_decomp_for_sector(me_terms, show_sector, sectors, show_macro, macro_vars, cumulative=False)
        plot_spatial_decomp_for_sector(me_terms, show_sector, sectors, show_macro, macro_vars, cumulative=True)
    
    # 4) Simulation scénarios: ΔX = X_s - X_base -> ΔZ
    dZ_by_scenario = {}
    contrib_by_scenario = {}
    
    for name, X_sc in X_scenarios.items():
        dX = (X_sc[macro_vars] - X_base[macro_vars]).copy()
        dZ = simulate_deltaZ(A_list, C_list, dX)
        dZ.columns = sectors
        dZ_by_scenario[name] = dZ
        
        # contributions par macro (optionnel mais super utile)
        contrib = contributions_by_macro(A_list, C_list, dX)
        for k in contrib:
            contrib[k].columns = sectors
        contrib_by_scenario[name] = contrib
    
    plot_scenario_paths(dZ_by_scenario, sectors, top_n=6, title="Comparaison scénarios: réponse ΔZ")
    
    if show_sector is not None:
        # affiche décomposition macro pour 1 scénario (le 1er) sur un secteur
        first_scen = list(contrib_by_scenario.keys())[0]
        plot_macro_contrib_for_sector(contrib_by_scenario[first_scen], show_sector, sectors,
                                      title=f"Décomposition macro ({first_scen})")
    
    return {
        "G0": G0, "B0": B0, "G_list": G_list, "Gamma_list": Gamma_list,
        "G0_inv": G0_inv, "A_list": A_list, "C_list": C_list,
        "ME_list": ME_list,
        "dZ_by_scenario": dZ_by_scenario,
        "contrib_by_scenario": contrib_by_scenario
    }